# 02 - Cleaning, Feature Engineering & Splitting

The logic lives in `src/` (so it is reusable and testable); this notebook runs it step by step and checks each step.

Steps: remove duplicates -> engineer features -> **split first** -> save to `data/processed/`.

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook runs from /notebooks or the repo root
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import TARGET, PROCESSED_DIR
from src.data_prep import load_raw, clean, split, save_splits, load_splits
from src.features import add_features, get_feature_columns

## 1. Remove duplicates

In [ ]:
raw = load_raw()
df = clean(raw)
print(f"rows before: {len(raw):,}  |  after: {len(df):,}  |  removed: {len(raw) - len(df):,}")
print(f"frauds before: {int(raw[TARGET].sum())}  |  after: {int(df[TARGET].sum())}")

## 2. Feature engineering

In [ ]:
df = add_features(df)
df[["Time", "hour", "Amount", "log_amount", "is_zero_amount"]].head()

| feature | why |
|---|---|
| `hour` | captures day/night cycle (relative hour; see `src/features.py`) |
| `log_amount` | `Amount` is heavily right-skewed; log makes it better behaved |
| `is_zero_amount` | zero-amount transactions are often card-validity probes |

Raw `Time` is **excluded from the model inputs**: it only encodes *where in this particular 2-day window* a transaction sits, which would not generalise to new data.

In [ ]:
feature_cols = get_feature_columns(df)
print(len(feature_cols), "model input features")
print(feature_cols)

## 3. Train / validation / test split

Done **before** any resampling or scaling, stratified so each split has the same fraud rate:

- **train (60%)** - fit models
- **validation (20%)** - compare models and choose the decision threshold
- **test (20%)** - used once at the very end for an honest estimate

In [ ]:
train, val, test = split(df)

check = pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "frauds": [int(train[TARGET].sum()), int(val[TARGET].sum()), int(test[TARGET].sum())],
}, index=["train", "val", "test"])
check["fraud_rate_%"] = (check["frauds"] / check["rows"] * 100).round(3)
check

In [ ]:
save_splits(train, val, test)
t, v, te = load_splits()
assert len(t) == len(train) and len(v) == len(val) and len(te) == len(test)
print("Saved and re-loaded OK ->", PROCESSED_DIR)

**Why split this way?** Resampling (SMOTE) must only ever see training data - applying it before splitting leaks synthetic copies of
test-set neighbours into training and produces unrealistically good scores. This is one of the most common mistakes in fraud projects.